<a href="https://colab.research.google.com/github/engmohammedhisham/flyrank-ml-internship-starter/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

**Method Choice:** Random Forest Classifier

**Why it fits our lane:**
Our task is to predict content performance decay (`trend_direction == 'down'`). The dataset features complex, non-linear interactions (e.g., a drop in `ctr` or `clicks_90d` might be critical only if the `avg_position` is already high). Linear or static rule-based systems fail to scale across these multi-dimensional boundary conditions. A Random Forest naturally detects these feature interactions, scales beautifully without requiring rigid normalization/scaling, handles minor outliers robustly, and yields clear global feature importances to help support editorial content refresh decisions.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

**Split Design:** Group-Based Validation (Grouping by `client_id`)

**Why this split is honest for our question:**
A standard random split would introduce severe **Data Leakage**. In production, our model will face an entirely new client with their unique site structures, domain authorities, and niche contexts. If we mix pages from the same `client_id` across both training and testing sets, the model will cheat by memorizing client-specific baseline traffic levels rather than learning general signals of content decay. Splitting by `client_id` guarantees that the test set measures true generalizability to unseen domains.

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [5]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# 1. Load the actual dataset
df = pd.read_csv('content_refresh_anonymized.csv')

# 2. Construct the binary target variable (1 if trend is downwards, 0 otherwise)
df['is_down'] = (df['trend_direction'] == 'down').astype(int)

# 3. Features selection (Continuous numeric search performance metrics)
features = ['search_volume', 'word_count', 'impressions_90d', 'clicks_90d', 'ctr', 'avg_position']
X = df[features].fillna(0)
y = df['is_down']
groups = df['client_id']

# 4. Implement Group Split (Using GroupKFold to isolate clients cleanly)
gkf = GroupKFold(n_splits=5)
train_idx, test_idx = next(gkf.split(X, y, groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

# --- WEEK-4 BASELINE: Predict decay if avg_position is poor (> 15) ---
y_pred_baseline = (X_test['avg_position'] > 15).astype(int)

# --- MACHINE LEARNING MODEL: Random Forest Classifier ---
model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
model.fit(X_train, y_train)

y_pred_model = model.predict(X_test)
y_prob_model = model.predict_proba(X_test)[:, 1]

# 5. Metric Collection Helper
def calculate_metrics(y_true, y_pred, y_prob=None):
    return {
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred, zero_division=0),
        'Recall': recall_score(y_true, y_pred, zero_division=0),
        'F1-Score': f1_score(y_true, y_pred, zero_division=0),
        'ROC-AUC': roc_auc_score(y_true, y_prob) if y_prob is not None else np.nan
    }

# 6. Generate Comparison Report
baseline_results = calculate_metrics(y_test, y_pred_baseline)
model_results = calculate_metrics(y_test, y_pred_model, y_prob_model)

performance_table = pd.DataFrame(
    [baseline_results, model_results],
    index=['Week-4 Baseline (Static Rule)', 'Random Forest Model']
)

print("### Model vs Baseline Performance Evaluation ###")
display(performance_table.round(4))

### Model vs Baseline Performance Evaluation ###


,Accuracy,Precision,Recall,F1-Score,ROC-AUC
Week-4 Baseline (Static Rule),0.3991,0.3315,0.2221,0.2660,NaN
Random Forest Model,0.5786,0.5479,0.8026,0.6512,0.6109


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [6]:
# Feature Importance Breakdown
importances = model.feature_importances_
indices = np.argsort(importances)[::-1]

print("### Model Feature Importance Ranking ###")
for i in range(X.shape[1]):
    print(f"{i + 1}. Feature '{features[indices[i]]}' (Importance: {importances[indices[i]]:.4f})")

# Error Breakdown (False Positives vs False Negatives)
fp = np.sum((y_pred_model == 1) & (y_test == 0))
fn = np.sum((y_pred_model == 0) & (y_test == 1))
print(f"\nError Analysis Summary: False Positives (FP) = {fp} | False Negatives (FN) = {fn}")

### Model Feature Importance Ranking ###
1. Feature 'impressions_90d' (Importance: 0.4774)
2. Feature 'avg_position' (Importance: 0.2740)
3. Feature 'word_count' (Importance: 0.1204)
4. Feature 'clicks_90d' (Importance: 0.0567)
5. Feature 'ctr' (Importance: 0.0562)
6. Feature 'search_volume' (Importance: 0.0152)

Error Analysis Summary: False Positives (FP) = 2275 | False Negatives (FN) = 678


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.